In [1]:
from transformers import AutoModelForSequenceClassification
import torch
dataset_name = "imdb"
tokenizer_checkpoint = "DeepWokLab/bert-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"


model = AutoModelForSequenceClassification.from_pretrained("DeepWokLab/bert-tiny")

print(model)
print(model.device)

/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/vol/bitbucket/dt822/adls/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepWokLab/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-1

## Build Mase Graph for BERT-tiny

Construct a `MaseGraph` from the loaded Hugging Face BERT-tiny model and render it to an SVG file.

In [2]:
from chop import MaseGraph

# Ensure classification metadata is set before graph tracing.
model.config.problem_type = "single_label_classification"

mg = MaseGraph(
    model,
    hf_input_names=["input_ids", "attention_mask", "labels"],
)

output_svg = "bert-tiny-mase-graph.svg"
mg.draw(output_svg)

print(f"Mase graph constructed and saved to: {output_svg}")

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.


Mase graph constructed and saved to: bert-tiny-mase-graph.svg


## Replace BERT-tiny Linear Layers with `QLinearPacked`

Import `QLinearPacked` from `mase_cuda.mxint8.linear` and define a transform pass that recursively swaps compatible `torch.nn.Linear` modules.

In [3]:
import torch
from chop import MaseGraph

try:
    from mase_cuda.mxint8.linear import QLinearPacked
except ImportError as exc:
    raise ImportError(
        "Could not import mase-cuda. Install it with: "
        "pip install 'git+https://github.com/DeepWok/mase-cuda@422ea09b82262da72b415237e3d0c0cf1cbb22b2'"
    ) from exc


def _best_group_size_for_linear(linear: torch.nn.Linear, preferred_group_size: int) -> int:
    constraints = [linear.weight.numel()]
    if linear.bias is not None:
        constraints.append(linear.bias.numel())

    for group_size in range(preferred_group_size, 0, -1):
        if all(value % group_size == 0 for value in constraints):
            return group_size
    return 1


def replace_linear_with_qlinearpacked_transform_pass(mg, pass_args=None):
    """Replace torch.nn.Linear modules in mg.model with QLinearPacked.

    pass_args:
        group_size (int): preferred group size for packing (default: 16).
    """
    pass_args = pass_args or {}
    preferred_group_size = int(pass_args.get("group_size", 16))
    replaced = []

    def _replace(module: torch.nn.Module, prefix: str = ""):
        for child_name, child in list(module.named_children()):
            fq_name = f"{prefix}.{child_name}" if prefix else child_name
            if isinstance(child, torch.nn.Linear):
                group_size = _best_group_size_for_linear(child, preferred_group_size)
                qlinear = QLinearPacked.build_from_linear(child, group_size=group_size)
                setattr(module, child_name, qlinear)
                replaced.append((fq_name, group_size, child.in_features, child.out_features))
            else:
                _replace(child, fq_name)

    _replace(mg.model)

    mg_quant = MaseGraph(
        mg.model,
        hf_input_names=["input_ids", "attention_mask", "labels"],
    )

    return mg_quant, {"replaced_linear_layers": replaced}


mg_qlinear, pass_info = replace_linear_with_qlinearpacked_transform_pass(
    mg,
    pass_args={"group_size": 16},
)

print(f"Replaced {len(pass_info['replaced_linear_layers'])} Linear layers with QLinearPacked")
for name, gs, in_f, out_f in pass_info["replaced_linear_layers"][:8]:
    print(f"- {name}: in={in_f}, out={out_f}, group_size={gs}")

mg_qlinear.draw("bert-tiny-qlinear-mase-graph.svg")
print("Saved graph: bert-tiny-qlinear-mase-graph.svg")

ImportError: Could not import mase-cuda. Install it with: pip install 'git+https://github.com/DeepWok/mase-cuda@422ea09b82262da72b415237e3d0c0cf1cbb22b2'